# Data Analysis

## Preparations

First of all, we read the raw results.

In [ ]:
import pandas as pd

import os

from pathlib import Path


ROOT = Path(os.getcwd())


def data_dir(root: Path) -> Path:
    return root / "data"


def raw_results(root: Path) -> Path:
    return data_dir(root) / "raw_results" / "raw_results.csv"


rr = pd.read_csv(
    raw_results(ROOT),
    low_memory=False,
)

Then, we drop columns that are actually not usable.

In [ ]:
COLS_OF_INTEREST = [
    "full_name_of_repo",
    "commit_sha",
    "path",
    "is_ccdc_event",
    "detected_channel",
    "created_at",
    "pushed_at",
    "updated_at",
    "date",
]

UNNECESSARY_COLS = []

for col in rr.columns:
    if col not in COLS_OF_INTEREST:
        UNNECESSARY_COLS.append(col)

rr.drop(columns=UNNECESSARY_COLS, inplace=True)

Next, we perform data conversion.

In [ ]:
def to_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series
    s = series.astype("string").str.strip().str.lower()
    mapping = {"true": True, "false": False}
    return s.map(mapping)


rr["is_ccdc_event"] = to_bool(rr["is_ccdc_event"])

rr["detected_channel"] = rr["detected_channel"].astype("string").fillna("")

DATETIME_COLS = ["created_at", "pushed_at", "updated_at", "date"]

for col in DATETIME_COLS:
    if col in rr.columns:
        rr[col] = pd.to_datetime(rr[col], errors="coerce", utc=True)


At this point, our results look like this:

In [ ]:
rr.head()

From here on, we work with a new Python variable/identifier.

In [ ]:
results = rr.copy()

Now, we remove the channels detected for the subjects that are **not** CCDC events.

In [ ]:
results["detected_channel"] = (
    results["detected_channel"]
        .where(results["is_ccdc_event"] == True, "")
)

Next, we transform the data frame: each row should be a unique subject – one row per subject.

In [ ]:
KEY_COLS = [
    "full_name_of_repo",
    "commit_sha",
    "path",
]

gb = results.groupby(KEY_COLS, dropna=False)


def agg_channels(x: pd.Series) -> tuple[str, ...]:
    vals = [
        v
        for v in x.astype("string").tolist()
        if isinstance(v, str) and v.strip() != ""
    ]
    return tuple(sorted(set(vals)))


agg = gb.agg(
    is_ccdc_event=("is_ccdc_event", "first"),
    detected_channels=("detected_channel", agg_channels),
)

for col in COLS_OF_INTEREST:
    if col not in KEY_COLS and col not in agg.columns and col != "detected_channel":
        agg[col] = gb[col].first()

results = agg.reset_index()

At this point, we start extracting further information based on the metadata already present in the data frame.

We define the first activity of a repository to be the first point in time providing evidence for project activity: either the time the repository was created on GitHub or the time of the first commit.

In [ ]:
results["first_activity"] = (
    results[["date", "created_at"]]
        .min(axis=1)
        .groupby(results["full_name_of_repo"], dropna=False)
        .transform("min")
)

The repository age is the simply the time passed since the repository's first activity.

In [ ]:
import numpy as np

results["repo_age"] = (
    (results["date"] - results["first_activity"]).to_numpy()
    / np.timedelta64(1, "D")
)

results.sort_values(["full_name_of_repo", "repo_age"], inplace=True)
results.reset_index(drop=True, inplace=True)

Based on the age of the repository at the time of a commit, we can assign the subjects to a set of age groups.

In [ ]:
AGE_GROUPS = [
    (0, 1, "0-1"),
    (1, 2, "1-2"),
    (2, 3, "2-3"),
    (3, 4, "3-4"),
    (4, 5, "4-5"),
    (5, 6, "5-6"),
    (6, 7, "6-7"),
    (7, 8, "7-8"),
    (8, 9, "8-9"),
    (9, 10, "9-10"),
    (10, 11, "10-11"),
    (11, 12, "11-12"),
    (12, 13, "12-13"),
    (13, 14, "13-14"),
    (14, 15, "14-15"),
    (15, 999, "15+"),
]


def assign_age_group(age_in_days: float) -> int | None:
    if pd.isna(age_in_days):
        return None
    for lo, hi, label in AGE_GROUPS:
        if lo * 365.25 <= age_in_days < hi * 365.25:
            return lo
    return None


results["age_group"] = results["repo_age"].apply(assign_age_group)

Later, we will group subjects by the year of their commits. Out of convenience, we introduce an extra column for that.

In [ ]:
results["year"] = results["date"].dt.year

Now, we create a dedicated data frame for the repository demographics.

In [ ]:
repos = (
    results
        .groupby("full_name_of_repo", as_index=False)
        .agg(
            created_at=("created_at", "first"),
            pushed_at=("pushed_at", "first"),
            updated_at=("updated_at", "first"),
            first_activity=("first_activity", "first"),
            first_subject_at=("date", "min"),
            last_subject_at=("date", "max"),
        )
)

This allows us to remove repository metadata from the results data frame.

In [ ]:
results.drop(
    columns=["created_at", "pushed_at", "updated_at", "first_activity"],
    inplace=True,
)

Since we introduced *first_activity*, consequently we introduce *last_activity*: the timestamp that is more recent, *pushed_at* or *updated_at*.

In [ ]:
repos["last_activity"] = repos[["pushed_at", "updated_at"]].max(axis=1)

Then we create columns for the age (in days), the age group, and the age group at the time of a repository's first subject.

In [ ]:

repos["age"] = (
    (repos["last_activity"] - repos["first_activity"]).to_numpy()
    / np.timedelta64(1, "D")
)

repos["age_group"] = repos["age"].apply(assign_age_group)

repos["days_until_first_subject"] = (
    (repos["first_subject_at"] - repos["first_activity"]).to_numpy()
    / np.timedelta64(1, "D")
)
repos["age_group_at_first_subject"] = repos["days_until_first_subject"].apply(assign_age_group)

repos.drop(columns=["days_until_first_subject"], inplace=True)
repos.sort_values("full_name_of_repo", ascending=False, inplace=True)
repos.reset_index(drop=True, inplace=True)

Our repository data frame looks like this:

In [ ]:
repos.head()

The base sample of repositories is from June 2019. Hence, we remove any subjects originating from repositories that were added to GitHub after June 2019.

In [ ]:
repos = repos.set_index("full_name_of_repo")
results = results.set_index("full_name_of_repo")

mask = repos["created_at"] >= "2019-07-01"

repos = repos.loc[~mask]
results = results.loc[results.index.isin(repos.index)]
repos.reset_index(inplace=True)
results.reset_index(inplace=True)

We make the path values case insensitive to only analyze how README and CONTRIBUTING files are distributed over time independent of a project maintainers opinion on whether or not a README/CONTRIBUTING file should be named in ALL CAPS.

In [ ]:
results["path"] = (
    results["path"]
    .str.split(".", n=1)
    .apply(lambda x: x[0].upper() + ("." + x[1] if len(x) > 1 else ""))
)

Since I do not know whether and how there are differences between CONTRIBUTING files and README files regarding what project maintainers decide to document, I decided to define a dedicated data frame for README files only. This allows for a more nuanced analysis later.

**However, originally I had included CONTRIBUTING files arguing that more and more projects these days point to their CONTRIBUTING files for communication channel relevant information. So, it makes sense, to keep the subjects that come from CONTRIBUTING files in the data set of the analysis.**

In [ ]:
# readme_results = results[results["path"].str.contains("README", case=False, na=False)]

We split the results/subjects in two: subjects before the base sample was drawn and subjects after June 2019. Why? Because if it was true that projects create CCDC events in their early years, not having any new projects for the time after June 2019 would create a bias in the data. How would we be able to analyze trends over time?

In [ ]:
results_before_july_2019 = results[results["date"] < "2019-07-01"]
results_since_july_2019 = results[results["date"] >= "2019-07-01"]

## Setup and Helper Functions

In [ ]:
from collections.abc import Callable, Sequence
from dataclasses import dataclass


MetricFn = Callable[[pd.DataFrame], object]


def _require_cols(g: pd.DataFrame, cols: Sequence[str]) -> None:
    missing = [c for c in cols if c not in g.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")


# ---------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------
def m_n_subjects(g: pd.DataFrame) -> int:
    return int(len(g))


def m_n_repos(g: pd.DataFrame) -> int:
    _require_cols(g, ["full_name_of_repo"])
    return int(g["full_name_of_repo"].nunique())


def m_n_commits(g: pd.DataFrame) -> int:
    _require_cols(g, ["commit_sha"])
    return int(g["commit_sha"].nunique())


def m_n_distinct_paths(g: pd.DataFrame) -> int:
    _require_cols(g, ["path"])
    return int(g["path"].nunique())


def m_positive_rate(g: pd.DataFrame) -> float:
    _require_cols(g, ["is_ccdc_event"])
    n = len(g)
    return np.nan if n == 0 else float(g["is_ccdc_event"].sum() / n)


def m_n_distinct_channels(g: pd.DataFrame) -> int:
    _require_cols(g, ["detected_channels"])
    return int(
        g["detected_channels"]
        .explode()
        .dropna()
        .nunique()
    )


# ---------------------------------------------------------------------
# Registry: metric_name -> (fn, required_columns)
# ---------------------------------------------------------------------
METRICS: dict[str, tuple[MetricFn, tuple[str, ...]]] = {
    "n_subjects": (m_n_subjects, ()),
    "n_repos": (m_n_repos, ("full_name_of_repo",)),
    "n_commits": (m_n_commits, ("commit_sha",)),
    "n_distinct_paths": (m_n_distinct_paths, ("path",)),
    "positive_rate": (m_positive_rate, ("is_ccdc_event",)),
    "n_distinct_channels": (
        m_n_distinct_channels,
        ("detected_channels",),
    ),
}

ALL_METRICS = (
    "n_subjects",
    "n_repos",
    "n_commits",
    "n_distinct_paths",
    "positive_rate",
    "n_distinct_channels",
)


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
@dataclass(frozen=True)
class SummarizeConfig:
    include_metrics: tuple[str, ...]
    group_keys: tuple[str, ...] = ()
    forbid_access_to_group_keys: bool = True
    drop_group_keys_from_frame: bool = True
    deny_columns: tuple[str, ...] = ()


# ---------------------------------------------------------------------
# Core summarization
# ---------------------------------------------------------------------
def summarize_subjects_configurable(
    g: pd.DataFrame,
    cfg: SummarizeConfig,
) -> pd.Series:

    g_eff = (
        g.drop(columns=list(cfg.group_keys), errors="ignore")
        if cfg.drop_group_keys_from_frame and cfg.group_keys
        else g
    )

    forbidden = set(cfg.deny_columns)
    if cfg.forbid_access_to_group_keys:
        forbidden |= set(cfg.group_keys)

    out: dict[str, object] = {}

    for name in cfg.include_metrics:
        if name not in METRICS:
            raise KeyError(
                f"Unknown metric: {name}. "
                f"Known: {sorted(METRICS)}"
            )

        fn, required = METRICS[name]

        illegal = [c for c in required if c in forbidden]
        if illegal:
            raise ValueError(
                f"Metric '{name}' requires forbidden columns {illegal}. "
                f"Forbidden: {sorted(forbidden)}"
            )

        out[name] = fn(g_eff)

    return pd.Series(out)

In [ ]:
def distribution_stats(
    s: pd.Series,
    *,
    prefix: str = "",
    dropna: bool = True,
) -> pd.Series:
    """
    Core descriptive stats for a numeric series, including boxplot whiskers/outliers.
    Works for both raw numeric data and 'counts per category' (value_counts output values).
    """
    if dropna:
        s = s.dropna()

    if s.empty:
        return pd.Series(dtype="float64")

    s = pd.to_numeric(s, errors="coerce").dropna()
    if s.empty:
        return pd.Series(dtype="float64")

    q1 = s.quantile(0.25)
    q2 = s.quantile(0.50)
    q3 = s.quantile(0.75)
    iqr = q3 - q1

    # Handle iqr=0 robustly (all values equal) without producing empty slices
    if pd.isna(iqr) or iqr == 0:
        lower_whisker = s.min()
        upper_whisker = s.max()
        n_outliers = 0
    else:
        lo = q1 - 1.5 * iqr
        hi = q3 + 1.5 * iqr
        lower_whisker = s[s >= lo].min()
        upper_whisker = s[s <= hi].max()
        n_outliers = int(((s < lower_whisker) | (s > upper_whisker)).sum())

    def k(name: str) -> str:
        return f"{prefix}{name}" if prefix else name

    return pd.Series({
        k("n"): int(len(s)),
        k("min"): float(s.min()),
        k("max"): float(s.max()),
        k("mean"): float(s.mean()),
        k("median"): float(q2),
        k("q1"): float(q1),
        k("q3"): float(q3),
        k("iqr"): float(iqr),
        k("lower_whisker"): float(lower_whisker),
        k("upper_whisker"): float(upper_whisker),
        k("n_outliers"): int(n_outliers),
    })


def boxplot_stats(s: pd.Series) -> pd.Series:
    # Now just a thin wrapper around the shared core
    return distribution_stats(s)


def value_counts_stats(vc: pd.Series) -> pd.Series:
    """
    Stats for a value_counts() result: distribution stats of the counts + concentration metrics.
    Expects vc to be sorted descending (as value_counts() returns by default).
    """
    vc = vc.dropna()
    if vc.empty:
        return pd.Series(dtype="float64")

    total = int(vc.sum())
    n_categories = int(len(vc))

    # shared distribution/boxplot stats over "counts per category"
    core = distribution_stats(vc, prefix="count_")

    # concentration / dominance (requires descending order)
    top_1 = int(vc.iloc[0])
    top_5_sum = int(vc.iloc[:5].sum())
    top_10_sum = int(vc.iloc[:10].sum())

    conc = pd.Series({
        "n_categories": n_categories,
        "total_count": total,
        "top_1_count": top_1,
        "top_1_share": (top_1 / total) if total else float("nan"),
        "top_5_share": (top_5_sum / total) if total else float("nan"),
        "top_10_share": (top_10_sum / total) if total else float("nan"),
    })

    return pd.concat([conc, core])

In [ ]:
from collections.abc import Iterable

def col_to_int(df: pd.DataFrame, int_cols: Iterable[str]) -> pd.DataFrame:
    cols = [c for c in int_cols if c in df.columns]
    df[cols] = df[cols].astype("Int64")
    return df


INT_COLS_OF_SUMMARY = [
    "n_subjects",
    "n_repos",
    "n_commits",
    "n_distinct_paths",
    "n_distinct_channels",
]

INT_COLS_OF_VC_STATS = [
    "n_categories",
    "total_count",
    "min_count",
    "max_count",
    "median_count",
    "q1_count",
    "q3_count",
    "iqr_count",
    "top_1_count",
]

In [ ]:
def vc_detected_channels(
    g: pd.DataFrame,
    *,
    drop_empty: bool = True,
) -> pd.Series:
    """
    Returns value_counts of channels for one group g.
    Assumes g["detected_channels"] contains iterables/tuples of channels, possibly empty () or NaN.
    """
    col: str = "detected_channels"
    s = g[col]

    # explode expects list-like; NaN stays NaN; () becomes empty -> drops on explode
    exploded = s.explode()

    # Optional cleanup
    exploded = exploded.dropna()
    exploded = exploded.astype("string")

    if drop_empty:
        exploded = exploded[exploded.str.strip() != ""]

    # value_counts -> counts per channel
    vc = exploded.value_counts(dropna=False)

    # Make the result stable/consistent
    vc.index.name = "channel"
    vc.name = "count"
    return vc

In [ ]:
import matplotlib as mpl

mpl.rcParams.update({
    # --- text / fonts ---
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,

    # --- lines ---
    "lines.linewidth": 1.2,
    "lines.markersize": 4,

    # --- axes / spines ---
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linewidth": 0.4,
    "grid.alpha": 0.3,

    # --- figure / export ---
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def set_size(width_mm=85, height_mm=60):
    w = width_mm / 25.4
    h = height_mm / 25.4
    return (w, h)

## Demographics

### Subject Count per Year

The following Code cell prepares a data frame called **time_results** that does not include subjects belonging to a year that has less subjects than the first quantile of the *subject count per year*.

In [ ]:
subject_count_per_year = (
    results
    .groupby("year")
    .size()
)

_to_be_removed = subject_count_per_year[subject_count_per_year < int(np.round(subject_count_per_year.quantile(0.25)))]

time_results = results[
    ~results["year"].isin(_to_be_removed.index)
]

In [ ]:
_to_be_removed

### Subject Count per Age Group

The following Code cell prepares a data frame called **project_results** that does not include subjects belonging to an age group that has less subjects than the first quantile of the *subject count per age group*.

In [ ]:
subject_count_per_age_group = (
    results
    .groupby("age_group")
    .size()
)

_to_be_removed = subject_count_per_age_group[subject_count_per_age_group < int(np.round(subject_count_per_age_group.quantile(0.25)))]

project_results = results[
    ~results["age_group"].isin(_to_be_removed.index)
]

### Our Results Grouped By …

#### Year – For A Repository-independent Perspective

In [ ]:
_results = time_results.copy()

_ccdc_events = time_results[time_results["is_ccdc_event"] == True]

_group_keys = ["year"]
cfg = SummarizeConfig(
    include_metrics=(
        "n_subjects",
        "n_repos",
        "positive_rate",
    ),
    group_keys=tuple(_group_keys),
)

_results = (
    _results
    .groupby(_group_keys)[["full_name_of_repo", "is_ccdc_event"]]
    .apply(lambda g: summarize_subjects_configurable(g, cfg))
)
_results = col_to_int(_results, INT_COLS_OF_SUMMARY)

cfg = SummarizeConfig(
    include_metrics=(
        "n_subjects",
        "n_repos",
        "n_distinct_channels",
    ),
    group_keys=tuple(_group_keys),
)
_ccdc_events = (
    _ccdc_events
    .groupby(_group_keys)[["full_name_of_repo", "detected_channels"]]
    .apply(lambda g: summarize_subjects_configurable(g, cfg))
)
_ccdc_events = col_to_int(_ccdc_events, INT_COLS_OF_SUMMARY)

_ccdc_events = _ccdc_events.rename(columns={"n_subjects": "n_ccdc_events"})
_ccdc_events = _ccdc_events.rename(columns={"n_repos": "n_pos_repos"})

_ccdc_events["n_ccdc_events_per_pos_repo"] = (
    _ccdc_events["n_ccdc_events"] / _ccdc_events["n_pos_repos"]
)

_ccdc_events["n_channels_per_pos_repo"] = (
    _ccdc_events["n_distinct_channels"] / _ccdc_events["n_pos_repos"]
)

_results = _results.join(
    _ccdc_events,
    how="left",
)

time_results_gb = _results.copy()
time_ccdc_events_gb = _ccdc_events.copy()

#### Age Group – For A Perspective On The Repository Life Cycle

In [ ]:
_results = project_results.copy()

_ccdc_events = project_results[project_results["is_ccdc_event"] == True]

_group_keys = ["age_group"]
cfg = SummarizeConfig(
    include_metrics=(
        "n_subjects",
        "n_repos",
        "positive_rate",
    ),
    group_keys=tuple(_group_keys),
)

_results = (
    _results
    .groupby(_group_keys)[["full_name_of_repo", "is_ccdc_event"]]
    .apply(lambda g: summarize_subjects_configurable(g, cfg))
)
_results = col_to_int(_results, INT_COLS_OF_SUMMARY)

cfg = SummarizeConfig(
    include_metrics=(
        "n_subjects",
        "n_repos",
        "n_distinct_channels",
    ),
    group_keys=tuple(_group_keys),
)
_ccdc_events = (
    _ccdc_events
    .groupby(_group_keys)[["full_name_of_repo", "detected_channels"]]
    .apply(lambda g: summarize_subjects_configurable(g, cfg))
)
_ccdc_events = col_to_int(_ccdc_events, INT_COLS_OF_SUMMARY)

_ccdc_events = _ccdc_events.rename(columns={"n_subjects": "n_ccdc_events"})
_ccdc_events = _ccdc_events.rename(columns={"n_repos": "n_pos_repos"})

_ccdc_events["n_ccdc_events_per_pos_repo"] = (
    _ccdc_events["n_ccdc_events"] / _ccdc_events["n_pos_repos"]
)

_ccdc_events["n_channels_per_pos_repo"] = (
    _ccdc_events["n_distinct_channels"] / _ccdc_events["n_pos_repos"]
)

_results = _results.join(
    _ccdc_events,
    how="left",
)

project_results_gb = _results.copy()
project_ccdc_events_gb = _ccdc_events.copy()

### Most Basic Demographics

What is the time range of the data?

In [ ]:
earliest_activity = repos["first_activity"].min()
most_recent_activity = repos["last_activity"].max()

print(f"Overall, the first (oldest) activity dates back to: {earliest_activity}.")
print(f"Again, overall, the last (most recent) activity dates back to: {most_recent_activity}.")

In [ ]:
years = pd.DataFrame({
    "year": range(
        earliest_activity.year,
        most_recent_activity.year + 1,
    )
})
years.set_index("year", inplace=True)

How many commits were processed?

In [ ]:
print(f"{results["commit_sha"].nunique()} commits were processed.")

How many subjects were processed and will be analyzed in this Jupyter Notebook?

In [ ]:
print(f"{len(results)} subjects were processed and will be analyzed here.")

Now, let's summarize the entire results data set:

In [ ]:
group_keys = []
cfg = SummarizeConfig(
    include_metrics=ALL_METRICS,
    group_keys=tuple(group_keys),
)

summary = summarize_subjects_configurable(results, cfg).to_frame().T
summary = col_to_int(summary, INT_COLS_OF_SUMMARY)
summary

### Path Demographics

Next, we group the subjects by path and create a summary for each group.

In [ ]:
group_keys = ["path"]

cfg = SummarizeConfig(
    include_metrics=(
        "n_subjects",
        "n_repos",
        "n_commits",
    ),
    group_keys=tuple(group_keys),
)

path_summaries = (
    results
        .groupby(group_keys)[["full_name_of_repo", "commit_sha"]]
        .apply(lambda g: summarize_subjects_configurable(g, cfg))
)

path_summaries = col_to_int(path_summaries, INT_COLS_OF_SUMMARY)
path_summaries.sort_values("n_subjects", ascending=False, inplace=True)

path_summaries_bj19 = (
    results_before_july_2019
        .groupby(group_keys)[["full_name_of_repo", "commit_sha"]]
        .apply(lambda g: summarize_subjects_configurable(g, cfg))
)

path_summaries_bj19 = col_to_int(path_summaries_bj19, INT_COLS_OF_SUMMARY)
path_summaries_bj19.sort_values("n_subjects", ascending=False, inplace=True)

path_summaries_sj19 = (
    results_since_july_2019
        .groupby(group_keys)[["full_name_of_repo", "commit_sha"]]
        .apply(lambda g: summarize_subjects_configurable(g, cfg))
)

path_summaries_sj19 = col_to_int(path_summaries_sj19, INT_COLS_OF_SUMMARY)
path_summaries_sj19.sort_values("n_subjects", ascending=False, inplace=True)

In [ ]:
path_summaries

In [ ]:
path_summaries_bj19

In [ ]:
path_summaries_sj19

What about the ratio of CONTRIBUTING.md files to README.md files? …

Now, we create a bar chart showing how many subjects belong to a path for each year of the time span:

In [ ]:
results_gb_year_path = (
    time_results
    .groupby(["year", "path"])
    .size()
    .reset_index(name="count")
)

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

sns.barplot(
    data=results_gb_year_path,
    x="year",
    y="count",
    hue="path"
)

plt.xlabel("Year")
plt.ylabel("Number of Subjects")
plt.xticks(rotation=45)
plt.legend(title="Path")

plt.tight_layout()
plt.show()

### Repository Demographics

Next, we take a look at the repository specific demographics…

#### Repository Count Overall

In [ ]:
repo_count = len(repos)

assert(results["full_name_of_repo"].nunique() == repo_count)
print(f"{repo_count} repositories contributed to the results.")

#### Repositories Come and Go…

Now, we create a table showing how many repositories were born in a year and how many repositories had their last activity in a year.

In [ ]:
_repos_copy = repos.copy()

_repos_copy["year_of_first_activity"] = _repos_copy["first_activity"].dt.year
_repos_copy["year_of_last_activity"] = _repos_copy["last_activity"].dt.year

_repos_come = (
    _repos_copy
    .groupby("year_of_first_activity")
    .size()
    .reset_index(name="count")
)

_repos_come.set_index("year_of_first_activity", drop=True, inplace=True)
_repos_come.rename_axis("year", inplace=True)

_repos_come = years.join(    
    _repos_come.rename_axis("year"),
    how="left"
)

_repos_come["count"] = _repos_come["count"].fillna(0).astype(int)
_repos_come = _repos_come.rename(columns={"count": "n_repos_come"})

_repos_go = (
    _repos_copy
    .groupby("year_of_last_activity")
    .size()
    .reset_index(name="count")
)

_repos_go.set_index("year_of_last_activity", drop=True, inplace=True)
_repos_go.rename_axis("year", inplace=True)

_repos_go = years.join(
    _repos_go.rename_axis("year"),
    how="left"
)

_repos_go["count"] = _repos_go["count"].fillna(0).astype(int)
_repos_go = _repos_go.rename(columns={"count": "n_repos_go"})

repos_come_and_go = _repos_come.join(
    _repos_go,
    how="left",
)

repos_come_and_go["n_dead"] = (
    repos_come_and_go["n_repos_go"]
    .cumsum()
    .shift(1, fill_value=0)
)

repos_come_and_go["n_alive"] = (
    repos_come_and_go["n_repos_come"].cumsum()
    - repos_come_and_go["n_dead"]
)

In [ ]:
repos_come_and_go

In summary, this approach ensures that:
- repositories contribute to the active count in the year they are created,
- and repositories that become inactive are only removed from the active population in the following year.

This results in a temporally consistent estimate of the number of active repositories over time.

#### Age Groups Today

What do the age groups look like Today?

In [ ]:
group_keys = ["age_group"]

cfg = SummarizeConfig(
    include_metrics=(
        "n_repos",
    ),
    group_keys=tuple(group_keys),
)

repos_per_age_group_today = (
    repos
    .groupby(group_keys)[["full_name_of_repo"]]
    .apply(lambda g: summarize_subjects_configurable(g, cfg))
)

In [ ]:
repos_per_age_group_today

#### At What Age Do Repositories Have Their First Subject?

In [ ]:
_group_keys = ["age_group_at_first_subject"]

_cfg = SummarizeConfig(
    include_metrics=(
        "n_repos",
    ),
    group_keys=tuple(_group_keys),
)

age_at_first_subject = (
    repos
    .groupby(_group_keys)[["full_name_of_repo"]]
    .apply(lambda g: summarize_subjects_configurable(g, _cfg))
)

age_at_first_subject["share"] = (
    age_at_first_subject["n_repos"] / repo_count
)

In [ ]:
age_at_first_subject

#### Survivors

In [ ]:
survivors = (
    repos
    .groupby("age_group")
    .size()
    .reset_index(name="count")
)

survivors.set_index("age_group", drop=True, inplace=True)

survivors["n_alive"] = survivors["count"][::-1].cumsum()[::-1]

survivors = survivors.rename(columns={"count": "n_dead"})

survivors["n_survivors"] = survivors["n_alive"] - survivors["n_dead"]

In [ ]:
survivors

#### Each Year, How Many Repositories Have A Subject?

In [ ]:
time_results_gb = time_results_gb.join(
    repos_come_and_go["n_alive"],
    how="left",
)

_active_repos_per_year = time_results_gb.copy()

_active_repos_per_year["share"] = (
    _active_repos_per_year["n_repos"] / _active_repos_per_year["n_alive"]
)

_active_repos_per_year["share_rolling_mean_3"] = (
    _active_repos_per_year["share"].rolling(3, center=True).mean()
)

fig, ax = plt.subplots(figsize=set_size(170, 120))

ax.plot(_active_repos_per_year.index, _active_repos_per_year["share"])
ax.plot(_active_repos_per_year.index, _active_repos_per_year["share_rolling_mean_3"], label="3-year moving average")

ax.set_xlabel("Year")
ax.set_ylabel("Share of Repositories with a Subject")

ax.legend(frameon=False, loc="best")
ax.margins(x=0.02)

plt.show()

#### Per Age Group: Subject Count Per Repository

Over the lifetime of a repo, how many subjects per active repo? How does that number evolve?

*-> How does the commitment RE project documentation evolve over the lifetime of a repository?*

In [ ]:
project_results_gb["n_subjects_per_active_repo"] = (
    project_results_gb["n_subjects"] / project_results_gb["n_repos"]
)

_df = project_results_gb.copy()

_df["n_subjects_per_active_repo_rolling_mean_3"] = (
    _df["n_subjects_per_active_repo"].rolling(3, center=True).mean()
)

fig, ax = plt.subplots(figsize=set_size(170, 120))

ax.plot(_df.index, _df["n_subjects_per_active_repo"])
ax.plot(_df.index, _df["n_subjects_per_active_repo_rolling_mean_3"], label="3-year moving average")

ax.set_xlabel("Age Group")
ax.set_ylabel("Number of Subjects Per Repository")

ax.legend(frameon=False, loc="best")
ax.margins(x=0.02)

plt.show()

In [ ]:
project_results_gb = project_results_gb.join(
    survivors["n_alive"],
    how="left",
)

_active_repos_per_age_group = project_results_gb.copy()

_active_repos_per_age_group["share"] = (
    _active_repos_per_age_group["n_repos"] / _active_repos_per_age_group["n_alive"]
)

_df = _active_repos_per_age_group.copy()

_df["share_rolling_mean_3"] = (
    _df["share"].rolling(3, center=True).mean()
)

fig, ax = plt.subplots(figsize=set_size(170, 120))

ax.plot(_df.index, _df["share"])
ax.plot(_df.index, _df["share_rolling_mean_3"], label="3-year moving average")

ax.set_xlabel("Age Group")
ax.set_ylabel("Share of Repositories with a Subject")

ax.legend(frameon=False, loc="best")
ax.margins(x=0.02)

plt.show()

#### Boxplot: Repository Age

In [ ]:
fig, ax = plt.subplots(figsize=set_size(170, 300))

ax.boxplot(
    np.round(repos["age"]),
    vert=True,
    widths=0.5,
    showmeans=True,
    meanline=False,
    whis=1.5,
)

ax.set_ylabel("Repository Age in Days")
ax.set_xticks([])

plt.show()

In [ ]:
# print(boxplot_stats(repos["age"]))

#### Boxplot: Subject Count Per Repository

In [ ]:
results_gb_repo = (
    results
    .groupby(["full_name_of_repo"])
    .size()
    .reset_index(name="count")
)

fig, ax = plt.subplots(figsize=set_size(170, 300))

ax.boxplot(
    results_gb_repo["count"],
    vert=True,
    widths=0.5,
    showmeans=True,
    meanline=False,
    whis=1.5,
)

ax.set_ylabel("Number of Subjects")
ax.set_xticks([])

plt.show()

In [ ]:
# print(boxplot_stats(results_gb_repo["count"]))

#### More Statistics…

Overall, how many CCDC events per repo (median)? How many channels per repo (median)?

In [ ]:
group_keys = ["full_name_of_repo"]

cfg = SummarizeConfig(
    include_metrics=(
        "positive_rate",
        "n_distinct_channels",
    ),
    group_keys=tuple(group_keys),
)

positive_rate_and_detected_channels_per_repo = (
    results
    .groupby(group_keys)[["is_ccdc_event", "detected_channels"]]
    .apply(lambda g: summarize_subjects_configurable(g, cfg))
)

fig, ax = plt.subplots(figsize=set_size(85, 120))

ax.boxplot(
    positive_rate_and_detected_channels_per_repo["positive_rate"],
    vert=True,
    widths=0.5,
    showmeans=True,
    meanline=False,
    whis=1.5,
)

ax.set_ylabel("CCDC Event Rate")
ax.set_xticks([])

plt.show()

In [ ]:
print(boxplot_stats(positive_rate_and_detected_channels_per_repo["positive_rate"]))

In [ ]:
fig, ax = plt.subplots(figsize=set_size(85, 120))

ax.boxplot(
    positive_rate_and_detected_channels_per_repo["n_distinct_channels"],
    vert=True,
    widths=0.5,
    showmeans=True,
    meanline=False,
    whis=1.5,
)

ax.set_ylabel("Number of Communication Channels")
ax.set_xticks([])

plt.show()

In [ ]:
print(boxplot_stats(positive_rate_and_detected_channels_per_repo["n_distinct_channels"]))

### CCDC Event Demographics

We filter our results to define a dedicated data frame that only includes CCDC events:

In [ ]:
ccdc_events = results[results["is_ccdc_event"] == True]

How many CCDC events are without any detected channel?

In [ ]:
empty_ccdc_events = ccdc_events[ccdc_events["detected_channels"] == ()]

print(f"There are {len(empty_ccdc_events)} CCDC events without any detected channel.")

_share = (
    len(empty_ccdc_events) / len(ccdc_events)
)

print(f"That is {_share*100:.2f}% of all CCDC events.")

When did these happen?

In [ ]:
_ccdc_events = ccdc_events.copy()

_ccdc_events["is_empty"] = _ccdc_events["detected_channels"] == ()

_empty_vs_non_empty = (
    _ccdc_events
    .groupby(["year", "is_empty"])
    .size()
    .reset_index(name="count")
)

import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

sns.barplot(
    data=_empty_vs_non_empty,
    x="year",
    y="count",
    hue="is_empty"
)

plt.xlabel("Year")
plt.ylabel("Number of CCDC Events")
plt.xticks(rotation=45)
plt.legend(title="The CCDC Event Is Empty")

plt.tight_layout()
plt.show()

## Repository-independent Perspective

#### CCDC Event Rate over the Years

Over the years, how did the CCDC event rate – the positive rate – evolve?

In [ ]:
_df = time_results_gb.copy()

_df["positive_rate_rolling_mean_3"] = (
    _df["positive_rate"].rolling(3, center=True).mean()
)

fig, ax = plt.subplots(figsize=set_size(170, 120))

ax.plot(_df.index, _df["positive_rate"])
ax.plot(_df.index, _df["positive_rate_rolling_mean_3"], label="3-year moving average")

ax.set_xlabel("Year")
ax.set_ylabel("CCDC Event Rate")

ax.legend(frameon=False, loc="best")
ax.margins(x=0.02)

plt.show()

#### Out of all Repositories that were alive, how many had a CCDC Event?

In [ ]:
time_results_gb["share_pos_repos"] = (
    time_results_gb["n_pos_repos"] / time_results_gb["n_alive"]
)

_df = time_results_gb.copy()

_df["share_rolling_mean_3"] = (
    _df["share_pos_repos"].rolling(3, center=True).mean()
)

fig, ax = plt.subplots(figsize=set_size(170, 120))

ax.plot(_df.index, _df["share_pos_repos"])
ax.plot(_df.index, _df["share_rolling_mean_3"], label="3-year moving average")

ax.set_xlabel("Year")
ax.set_ylabel("Share of Repositories with a CCDC Event")

ax.legend(frameon=False, loc="best")
ax.margins(x=0.02)

plt.show()

I think it is fascinating that the CCDC event rate is relatively stable over the years, while the share of repositories that have a CCDC event is dropping drastically… Having said that, let's also plot the number of CCDC events a positive repositories is responsible for, on average, per year.

In [ ]:
_df = time_results_gb.copy()

_df["n_ccdc_events_per_pos_repo_rolling_mean_3"] = (
    _df["n_ccdc_events_per_pos_repo"].rolling(3, center=True).mean()
)

fig, ax = plt.subplots(figsize=set_size(170, 120))

ax.plot(_df.index, _df["n_ccdc_events_per_pos_repo"])
ax.plot(_df.index, _df["n_ccdc_events_per_pos_repo_rolling_mean_3"], label="3-year moving average")

ax.set_xlabel("Year")
ax.set_ylabel("Number of CCDC Events per Positive Repository")

ax.legend(frameon=False, loc="best")
ax.margins(x=0.02)

plt.show()

#### Channel Diversity

In [ ]:
_df = time_results_gb.copy()

_df["rolling_mean_3"] = (
    _df["n_distinct_channels"].rolling(3, center=True).mean()
)

fig, ax = plt.subplots(figsize=set_size(170, 120))

ax.plot(_df.index, _df["n_distinct_channels"])
ax.plot(_df.index, _df["rolling_mean_3"], label="3-year moving average")

ax.set_xlabel("Year")
ax.set_ylabel("Number of Distinct Communication Channels")

ax.legend(frameon=False, loc="best")
ax.margins(x=0.02)
ax.set_ylim(17, 34)

import matplotlib.ticker as ticker

ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.yaxis.set_major_locator(ticker.MultipleLocator(2))

plt.show()

#### Communication Channel Count

We introduce a matrix that has a column for each channel and lists the channel count for a channel for a year.

In [ ]:
cc_count_matrix = (
    results
    .explode("detected_channels")
    .groupby(["year", "detected_channels"])
    .size()
    .unstack(fill_value=0)
)

cc_count_matrix.columns.name = None
cc_count_matrix = cc_count_matrix.loc[:, cc_count_matrix.iloc[-1].sort_values(ascending=False).index]
# cc_count_matrix = cc_count_matrix[cc_count_matrix.sum().sort_values(ascending=False).index]

##### Heatmap

Now, we plot a heatmap for the channel count matrix.

In [ ]:
fig, ax = plt.subplots(figsize=set_size(255, 140))

im = ax.imshow(cc_count_matrix, aspect="auto")

ax.set_xlabel("Communication Channel")
ax.set_ylabel("Year")

ax.set_xticks(range(len(cc_count_matrix.columns)))
ax.set_xticklabels(cc_count_matrix.columns, rotation=90)

ax.set_yticks(range(len(cc_count_matrix.index)))
ax.set_yticklabels(cc_count_matrix.index)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Number of Subjects")

plt.tight_layout()
plt.show()

##### Line Plot

Let's also plot how the count of a certain set of channels has evolved over the years.

In [ ]:
from matplotlib.ticker import MaxNLocator

CHANNELS_OF_INTEREST = [
    "github_issues",
    "issues",
    "mailing_list",
    "gitter",
    "github_discussions",
]

_df = (
    cc_count_matrix[[*CHANNELS_OF_INTEREST]]
)

_df = _df.loc[_df.index >= 2013]

fig, ax = plt.subplots(figsize=set_size(170, 85))

for ghc in CHANNELS_OF_INTEREST:
    ax.plot(_df.index, _df[ghc], label=ghc)

ax.xaxis.set_major_locator(
    MaxNLocator(integer=True)
)

ax.set_xlabel("Year")
ax.set_ylabel("Count")

ax.legend(frameon=False, loc="best")
ax.margins(x=0.02)
ax.set_ylim(0, 65)

import matplotlib.ticker as ticker

ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.yaxis.set_major_locator(ticker.MultipleLocator(10))

plt.show()

#### Channels per CCDC Event

In [ ]:
_ccdc_events = time_results[time_results["is_ccdc_event"] == True]

_vc_detected_channels_per_year = (
    _ccdc_events
    .groupby("year")[["detected_channels"]]
    .apply(vc_detected_channels)
)

_vc_stats_per_year = (
    _vc_detected_channels_per_year
    .reset_index()
    .groupby("year")["count"]
    .apply(value_counts_stats)
).unstack(fill_value=0)
_vc_stats_per_year = col_to_int(_vc_stats_per_year, INT_COLS_OF_VC_STATS)

vc_detected_channels_per_year = _vc_detected_channels_per_year.reset_index()

cc_count_per_year = _vc_stats_per_year[["total_count"]].join(
    time_results_gb["n_ccdc_events"],
    how="left",
)

cc_count_per_year["ratio"] = (
    cc_count_per_year["total_count"] / cc_count_per_year["n_ccdc_events"]
)

In [ ]:
cc_count_per_year

In [ ]:
_df = cc_count_per_year.copy()

_df["ratio_rolling_mean_3"] = (
    _df["ratio"].rolling(3, center=True).mean()
)

fig, ax = plt.subplots(figsize=set_size(170, 120))

ax.plot(_df.index, _df["ratio"])
ax.plot(_df.index, _df["ratio_rolling_mean_3"], label="3-year moving average")

ax.set_xlabel("Year")
ax.set_ylabel("Avg. Number of Detected Channels per CCDC Event")

ax.legend(frameon=False, loc="best")
ax.margins(x=0.02)
ax.set_ylim(0, 3)

import matplotlib.ticker as ticker

ax.xaxis.set_major_locator(ticker.MultipleLocator(1))

plt.show()

#### Top 5 Channels

In [ ]:
top_5_per_year = (
    vc_detected_channels_per_year
        .sort_values(["year", "count"], ascending=[True, False])
        .groupby("year", dropna=False, sort=False)
        .head(5)
        .reset_index(drop=True)
)

How many channels have ever made it into the top 5 of year?

In [ ]:
print(f"Answer: {top_5_per_year["channel"].nunique()}")

# print(f"These are:\n{top_5_per_year["channel"].unique()}")

In [ ]:
top_5_per_year[top_5_per_year["year"] == 2016]

In [ ]:
top_5_per_year[top_5_per_year["year"] == 2013]

In [ ]:
top_5_per_year[top_5_per_year["year"] == 2019]

In [ ]:
top_5_per_year[top_5_per_year["year"] == 2022]

## Repository Life Cycle Perspective

#### CCDC Event Rate

In the lifetime of a repository, how has the CCDC event rate evolved over time?

In [ ]:
from matplotlib.ticker import MaxNLocator

_df = project_results_gb.copy()

_df["positive_rate_rolling_mean_3"] = (
    _df["positive_rate"].rolling(3, center=True).mean()
)

fig, ax = plt.subplots(figsize=set_size(170, 120))

ax.plot(_df.index, _df["positive_rate"])
ax.plot(_df.index, _df["positive_rate_rolling_mean_3"], label="3-year moving average")

ax.xaxis.set_major_locator(
    MaxNLocator(integer=True)
)

ax.set_xlabel("Age Group")
ax.set_ylabel("CCDC Event Rate")

ax.legend(frameon=False, loc="best")
ax.margins(x=0.02)

plt.show()

#### The Likelihood of a Repository having a CCDC Event

In [ ]:
project_results_gb["share_pos_repos"] = (
    project_results_gb["n_pos_repos"] / project_results_gb["n_alive"]
)

_df = project_results_gb.copy()

_df["share_rolling_mean_3"] = (
    _df["share_pos_repos"].rolling(3, center=True).mean()
)

fig, ax = plt.subplots(figsize=set_size(170, 120))

ax.plot(_df.index, _df["share_pos_repos"])
ax.plot(_df.index, _df["share_rolling_mean_3"], label="3-year moving average")

ax.set_xlabel("Age Group")
ax.set_ylabel("Share of Repositories with a CCDC Event")

ax.legend(frameon=False, loc="best")
ax.margins(x=0.02)

plt.show()

#### Subject Count and CCDC Event Count

In [ ]:
_df = project_results_gb.copy()

fig, ax = plt.subplots(figsize=set_size(170, 120))

ax.plot(_df.index, _df["n_subjects"], label="Subject Count")
ax.plot(_df.index, _df["n_ccdc_events"], label="CCDC Event Count")

ax.set_xlabel("Age Group (Year)")
ax.set_ylabel("Count")

ax.legend(frameon=False, loc="best")
ax.margins(x=0.02)

plt.show()

#### Channel Diversity

In [ ]:
_df = project_results_gb.copy()

fig, ax = plt.subplots(figsize=set_size(170, 120))

ax.plot(_df.index, _df["n_distinct_channels"])

ax.set_xlabel("Age Group")
ax.set_ylabel("Number of Distinct Communication Channels")

ax.margins(x=0.02)
ax.set_ylim(17, 34)

import matplotlib.ticker as ticker

ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.yaxis.set_major_locator(ticker.MultipleLocator(2))

plt.show()

*Das zeigt eigentlich ziemlich deutlich, dass die Channel-Menge innerhalb eines Projektes nicht über Zeit zunimmt, sondern gleich bleibt oder abnimmt. Oder?*

#### Number Of Distinct Channels Per Positive Repository

In [ ]:
_df = project_results_gb.copy()

fig, ax = plt.subplots(figsize=set_size(170, 120))

ax.plot(_df.index, _df["n_channels_per_pos_repo"])

ax.set_xlabel("Age Group")
ax.set_ylabel("Number of Distinct Communication Channels")

ax.margins(x=0.02)
ax.set_ylim(0, 2)

import matplotlib.ticker as ticker

ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
# ax.yaxis.set_major_locator(ticker.MultipleLocator(2))

plt.show()

*Dadurch, dass positive Projekte schneller verschwinden, als die Channel Diversität abnimmt, kommen mehr distinct channels auf ein einzelnen positives Projekt – klar.*

#### Channels per CCDC Event

In [ ]:
_ccdc_events = project_results[project_results["is_ccdc_event"] == True]

_vc_detected_channels_per_age_group = (
    _ccdc_events
    .groupby("age_group")[["detected_channels"]]
    .apply(vc_detected_channels)
)

_vc_stats_per_age_group = (
    _vc_detected_channels_per_age_group
    .reset_index()
    .groupby("age_group")["count"]
    .apply(value_counts_stats)
).unstack(fill_value=0)
_vc_stats_per_age_group = col_to_int(_vc_stats_per_age_group, INT_COLS_OF_VC_STATS)

vc_detected_channels_per_age_group = _vc_detected_channels_per_age_group.reset_index()

cc_count_per_age_group = _vc_stats_per_age_group[["total_count"]].join(
    project_results_gb["n_ccdc_events"],
    how="left",
)

cc_count_per_age_group["ratio"] = (
    cc_count_per_age_group["total_count"] / cc_count_per_age_group["n_ccdc_events"]
)

In [ ]:
cc_count_per_age_group

In [ ]:
_df = cc_count_per_age_group.copy()

_df["ratio_rolling_mean_3"] = (
    _df["ratio"].rolling(3, center=True).mean()
)

fig, ax = plt.subplots(figsize=set_size(170, 120))

ax.plot(_df.index, _df["ratio"])
ax.plot(_df.index, _df["ratio_rolling_mean_3"], label="3-year moving average")

ax.set_xlabel("Age Group")
ax.set_ylabel("Avg. Number of Detected Channels per CCDC Event")

ax.legend(frameon=False, loc="best")
ax.margins(x=0.02)
ax.set_ylim(0, 3)

import matplotlib.ticker as ticker

ax.xaxis.set_major_locator(ticker.MultipleLocator(1))

plt.show()

#### Top 5 Channels

In [ ]:
_top_5_per_age_group = (
    vc_detected_channels_per_age_group
        .sort_values(["age_group", "count"], ascending=[True, False])
        .groupby("age_group", dropna=False, sort=False)
        .head(5)
        .reset_index(drop=True)
)

How many channels have ever made it into the top 5 of an age group?

In [ ]:
print(f"Answer: {_top_5_per_age_group["channel"].nunique()}")

# print(f"These are:\n{_top_5_per_age_group["channel"].unique()}")

##### First Year (Age Group 0)

In [ ]:
_top_5_per_age_group[_top_5_per_age_group["age_group"] == 0]

##### 4th Year (Age Group 3)

In [ ]:
_top_5_per_age_group[_top_5_per_age_group["age_group"] == 3]

##### 7th Year (Age Group 8)

In [ ]:
_top_5_per_age_group[_top_5_per_age_group["age_group"] == 8]

##### 10th Year (Age Group 9)

In [ ]:
_top_5_per_age_group[_top_5_per_age_group["age_group"] == 9]